In [1]:
!pip install langdetect textblob pysentimiento nltk -q
print("✓ Listo")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 24.7 MB/s eta 0:00:00
✓ Listo


In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from langdetect import detect, DetectorFactory
from textblob import TextBlob
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad en detección de idioma
DetectorFactory.seed = 42

# Cargar dataset crudo
from google.colab import files
uploaded = files.upload()

import io
nombre_archivo = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]))

print(f"✓ Dataset cargado: {df_raw.shape}")
print(f"\nColumnas: {list(df_raw.columns)}")
print(f"\nPrimeras filas:")
display(df_raw.head(3))

Saving tripadvisor_ecuador_raw_20260420.csv to tripadvisor_ecuador_raw_20260420.csv
✓ Dataset cargado: (43, 8)

Columnas: ['destino', 'categoria', 'region', 'titulo', 'texto', 'rating', 'fecha_raw', 'fecha_extraccion']

Primeras filas:


,destino,categoria,region,titulo,texto,rating,fecha_raw,fecha_extraccion
0,Galapagos,Naturaleza,Costa/Insular,Once in a lifetime experience,Visiting the Galapagos was the most extraordin...,5.0,January 2024,2026-04-20
1,Galapagos,Naturaleza,Costa/Insular,Naturaleza incomparable,Las Galápagos superaron todas mis expectativas...,5.0,Febrero 2024,2026-04-20
2,Galapagos,Naturaleza,Costa/Insular,Too expensive for what it offers,"The islands are beautiful, no doubt about that...",3.0,March 2024,2026-04-20


In [3]:
print("=" * 55)
print("INSPECCIÓN INICIAL DEL DATASET")
print("=" * 55)

print(f"\nDimensiones: {df_raw.shape[0]} reseñas × {df_raw.shape[1]} columnas")

print(f"\nDistribución por destino:")
print(df_raw.groupby('destino').size().sort_values(ascending=False).to_string())

print(f"\nDistribución por rating:")
print(df_raw['rating'].value_counts().sort_index().to_string())

print(f"\nNulos por columna:")
print(df_raw.isnull().sum().to_string())

print(f"\nEjemplo de texto:")
print(f"  Título : {df_raw['titulo'].iloc[0]}")
print(f"  Texto  : {df_raw['texto'].iloc[0][:120]}...")

INSPECCIÓN INICIAL DEL DATASET

Dimensiones: 43 reseñas × 8 columnas

Distribución por destino:
destino
Galapagos    8
Quito        8
Cuenca       6
Banos        5
Guayaquil    4
Mindo        4
Montanita    4
Otavalo      4

Distribución por rating:
rating
2.0     6
3.0     9
4.0    12
5.0    16

Nulos por columna:
destino             0
categoria           0
region              0
titulo              0
texto               0
rating              0
fecha_raw           0
fecha_extraccion    0

Ejemplo de texto:
  Título : Once in a lifetime experience
  Texto  : Visiting the Galapagos was the most extraordinary experience of my life. The wildlife is completely fearless, sea lions ...


In [4]:
def limpiar_texto(texto):
    """
    Limpieza estándar para NLP:
    - Elimina caracteres especiales y URLs
    - Normaliza espacios
    - Conserva puntuación básica (útil para análisis de sentimiento)
    """
    if not isinstance(texto, str) or texto.strip() == '':
        return ''
    # Eliminar URLs
    texto = re.sub(r'http\S+|www\S+', '', texto)
    # Eliminar caracteres especiales excepto letras, números y puntuación básica
    texto = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ0-9\s.,!?\'"-]', ' ', texto)
    # Normalizar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


df = df_raw.copy()

# Limpiar título y texto
df['titulo_limpio'] = df['titulo'].apply(limpiar_texto)
df['texto_limpio']  = df['texto'].apply(limpiar_texto)

# Texto completo = título + texto para análisis
df['texto_completo'] = (
    df['titulo_limpio'] + '. ' + df['texto_limpio']
).str.strip('. ')

# Longitud del texto (útil para filtrar reseñas vacías)
df['longitud_texto'] = df['texto_limpio'].apply(len)

print(f"✓ Limpieza completada")
print(f"  Reseñas con texto vacío: {(df['longitud_texto'] == 0).sum()}")
print(f"  Longitud promedio: {df['longitud_texto'].mean():.0f} caracteres")
print(f"  Longitud mínima:   {df['longitud_texto'].min()} caracteres")
print(f"  Longitud máxima:   {df['longitud_texto'].max()} caracteres")

✓ Limpieza completada
  Reseñas con texto vacío: 0
  Longitud promedio: 257 caracteres
  Longitud mínima:   225 caracteres
  Longitud máxima:   285 caracteres


In [5]:
def detectar_idioma(texto):
    """Detecta idioma del texto. Retorna 'es', 'en', u 'otro'."""
    if not isinstance(texto, str) or len(texto) < 15:
        return 'desconocido'
    try:
        idioma = detect(texto)
        if idioma == 'es':
            return 'es'
        elif idioma == 'en':
            return 'en'
        else:
            return 'otro'
    except Exception:
        return 'desconocido'

df['idioma'] = df['texto_completo'].apply(detectar_idioma)

print("✓ Detección de idioma completada")
print(f"\nDistribución de idiomas:")
dist = df['idioma'].value_counts()
for idioma, count in dist.items():
    pct = count / len(df) * 100
    print(f"  {idioma}: {count} reseñas ({pct:.1f}%)")

print(f"\nIdioma por destino:")
print(df.groupby(['destino','idioma']).size().unstack(fill_value=0).to_string())

✓ Detección de idioma completada

Distribución de idiomas:
  es: 24 reseñas (55.8%)
  en: 19 reseñas (44.2%)

Idioma por destino:
idioma     en  es
destino          
Banos       2   3
Cuenca      3   3
Galapagos   4   4
Guayaquil   2   2
Mindo       2   2
Montanita   2   2
Otavalo     1   3
Quito       3   5


In [6]:
# Asegurar que rating sea numérico
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Clasificación del rating en categorías
def clasificar_rating(r):
    if pd.isna(r):
        return 'sin_rating'
    elif r >= 4.0:
        return 'positivo'
    elif r == 3.0:
        return 'neutro'
    else:
        return 'negativo'

df['sentimiento_rating'] = df['rating'].apply(clasificar_rating)

# Normalizar fecha
df['fecha_raw'] = df['fecha_raw'].fillna('Desconocida')

# Año aproximado extraído de la fecha textual
def extraer_anio(fecha_str):
    match = re.search(r'20\d{2}', str(fecha_str))
    return int(match.group()) if match else None

df['anio'] = df['fecha_raw'].apply(extraer_anio)

print("✓ Variables derivadas creadas")
print(f"\nDistribución por sentimiento_rating:")
print(df['sentimiento_rating'].value_counts().to_string())

print(f"\nRating promedio por destino:")
print(df.groupby('destino')['rating'].mean()
        .sort_values(ascending=False)
        .round(2).to_string())

✓ Variables derivadas creadas

Distribución por sentimiento_rating:
sentimiento_rating
positivo    28
neutro       9
negativo     6

Rating promedio por destino:
destino
Otavalo      4.25
Mindo        4.25
Cuenca       4.17
Banos        4.00
Galapagos    4.00
Quito        3.75
Guayaquil    3.25
Montanita    3.25


In [7]:
n_antes = len(df)

# Eliminar reseñas con texto demasiado corto (menos de 30 caracteres)
df = df[df['longitud_texto'] >= 30].copy()

# Eliminar idiomas no identificados si son muy pocos
df = df[df['idioma'] != 'desconocido'].copy()

# Resetear índice
df = df.reset_index(drop=True)

n_despues = len(df)
print(f"✓ Filtros de calidad aplicados")
print(f"  Antes:  {n_antes} reseñas")
print(f"  Después: {n_despues} reseñas")
print(f"  Eliminadas: {n_antes - n_despues}")
print(f"\nDataset final: {df.shape}")

✓ Filtros de calidad aplicados
  Antes:  43 reseñas
  Después: 43 reseñas
  Eliminadas: 0

Dataset final: (43, 15)


In [8]:
nombre_procesado = 'tripadvisor_ecuador_procesado.csv'
df.to_csv(nombre_procesado, index=False, encoding='utf-8-sig')

print(f"✓ Dataset procesado guardado: {nombre_procesado}")
print(f"\nResumen final:")
print(f"  Total reseñas:    {len(df)}")
print(f"  Destinos:         {df['destino'].nunique()}")
print(f"  En español:       {(df['idioma']=='es').sum()}")
print(f"  En inglés:        {(df['idioma']=='en').sum()}")
print(f"  Rating promedio:  {df['rating'].mean():.2f} / 5.0")
print(f"  Columnas finales: {list(df.columns)}")

from google.colab import files
files.download(nombre_procesado)
print("✓ Descargado — guárdalo en data/processed/")

✓ Dataset procesado guardado: tripadvisor_ecuador_procesado.csv

Resumen final:
  Total reseñas:    43
  Destinos:         8
  En español:       24
  En inglés:        19
  Rating promedio:  3.88 / 5.0
  Columnas finales: ['destino', 'categoria', 'region', 'titulo', 'texto', 'rating', 'fecha_raw', 'fecha_extraccion', 'titulo_limpio', 'texto_limpio', 'texto_completo', 'longitud_texto', 'idioma', 'sentimiento_rating', 'anio']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Descargado — guárdalo en data/processed/
